# 05. Model comparison

Здесь мы делаем минимальное сравнение трёх baseline-моделей:
1. `Black-76 + hv_21d`
2. `Black-76 + hv_63d`
3. `Black-76 + garch_vol`

Цель - быстро понять, какая версия volatility input даёт наименьшие ошибки в среднем и как модели ведут себя в разных volatility regimes.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
project_root = Path('/Users/maria/Desktop/Code/HSE/COURSEBOOK')
baseline_path = project_root / 'results/model_pricing_baseline.parquet'
garch_path = project_root / 'results/model_pricing_garch.parquet'
results_dir = project_root / 'results'

baseline = pd.read_parquet(baseline_path)
garch = pd.read_parquet(garch_path)

baseline['TRADEDATE'] = pd.to_datetime(baseline['TRADEDATE'], errors='coerce')
garch['TRADEDATE'] = pd.to_datetime(garch['TRADEDATE'], errors='coerce')

baseline.shape, garch.shape

## Comparison setup

Для всех моделей считаем одинаковые метрики качества:
- `N`
- `MAE`
- `RMSE`
- `mean_error`
- `median_abs_error`

Это позволяет честно сравнить, какой вариант volatility input лучше воспроизводит рыночные цены.

In [ ]:
models = {
    'Black-76 + hv_21d': {
        'df': baseline,
        'model_col': 'model_price_hv_21d',
        'error_col': 'error_hv_21d',
        'abs_col': 'abs_error_hv_21d',
        'sq_col': 'squared_error_hv_21d',
    },
    'Black-76 + hv_63d': {
        'df': baseline,
        'model_col': 'model_price_hv_63d',
        'error_col': 'error_hv_63d',
        'abs_col': 'abs_error_hv_63d',
        'sq_col': 'squared_error_hv_63d',
    },
    'Black-76 + garch_vol': {
        'df': garch,
        'model_col': 'model_price_garch',
        'error_col': 'error_garch',
        'abs_col': 'abs_error_garch',
        'sq_col': 'squared_error_garch',
    },
}


def calc_metrics(frame, model_col, error_col, abs_col, sq_col):
    sub = frame.loc[frame[model_col].notna()].copy()
    return {
        'N': int(len(sub)),
        'MAE': float(sub[abs_col].mean()),
        'RMSE': float(np.sqrt(sub[sq_col].mean())),
        'mean_error': float(sub[error_col].mean()),
        'median_abs_error': float(sub[abs_col].median()),
    }


def calc_by_vol_regime(frame, model_col, error_col, abs_col, sq_col):
    sub = frame.loc[frame[model_col].notna()].copy()
    out = (
        sub.groupby('vol_regime', dropna=False, observed=False)
        .agg(
            N=(error_col, 'size'),
            MAE=(abs_col, 'mean'),
            mean_error=(error_col, 'mean'),
            median_abs_error=(abs_col, 'median'),
        )
        .reset_index()
    )
    rmse = (
        sub.groupby('vol_regime', dropna=False, observed=False)[sq_col]
        .apply(lambda s: float(np.sqrt(np.mean(s))))
        .reset_index(name='RMSE')
    )
    return out.merge(rmse, on='vol_regime', how='left')

## Overall comparison

Сначала сравниваем модели на всей выборке. Это даёт главный ответ: какой volatility input в среднем лучше всего воспроизводит рыночную цену опциона.

In [ ]:
summary_rows = []
for model_name, spec in models.items():
    row = {'model': model_name}
    row.update(calc_metrics(spec['df'], spec['model_col'], spec['error_col'], spec['abs_col'], spec['sq_col']))
    summary_rows.append(row)

comparison_summary = pd.DataFrame(summary_rows).sort_values('MAE').reset_index(drop=True)
comparison_summary

**Интерпретация.** По общему качеству лучший результат здесь показывает `Black-76 + hv_63d`: у него самые низкие `MAE` и `RMSE`. `GARCH` в текущем MVP не даёт явного улучшения относительно historical-vol baseline, а `hv_21d` и `garch_vol` выглядят очень близко друг к другу.

## Comparison by volatility regime

Теперь посмотрим, сохраняется ли это преимущество в разных volatility regimes. Особенно важно понять, растут ли ошибки в `high_vol` и помогает ли GARCH в стрессовых периодах.

In [ ]:
vol_tables = []
for model_name, spec in models.items():
    table = calc_by_vol_regime(spec['df'], spec['model_col'], spec['error_col'], spec['abs_col'], spec['sq_col'])
    table['model'] = model_name
    vol_tables.append(table)

comparison_by_vol = pd.concat(vol_tables, ignore_index=True)
comparison_by_vol

**Интерпретация.** Ошибки действительно растут в `high_vol` regime у всех моделей. `hv_63d` особенно хорошо выглядит в `low_vol`, но в `high_vol` его преимущество уже не так велико. В текущем MVP `GARCH` не показывает уверенного превосходства над `historical volatility` даже в high-volatility state.

## Simple plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(comparison_summary['model'], comparison_summary['MAE'])
axes[0].set_title('Overall MAE by model')
axes[0].set_ylabel('MAE')
axes[0].tick_params(axis='x', rotation=20)

plot_df = comparison_by_vol.dropna(subset=['vol_regime']).copy()
pivot = plot_df.pivot(index='vol_regime', columns='model', values='MAE')
pivot.plot(kind='bar', ax=axes[1])
axes[1].set_title('MAE by model and vol_regime')
axes[1].set_ylabel('MAE')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## Final short conclusion

- По overall quality лучший результат в этом MVP показывает `Black-76 + hv_63d`.
- `GARCH` не даёт явного улучшения относительно `historical volatility` baseline.
- У всех моделей ошибки растут в `high_vol` regime, то есть именно стрессовые периоды остаются главным источником model misspecification.
- Практический вывод: в курсовой разумно оставить `hv_63d` как лучший простой baseline, а `GARCH` - как более сложную альтернативу, которая пока не даёт явного выигрыша.

## Save outputs

In [ ]:
summary_path = results_dir / 'model_comparison_summary.csv'
vol_path = results_dir / 'model_comparison_by_vol_regime.csv'

comparison_summary.to_csv(summary_path, index=False)
comparison_by_vol.to_csv(vol_path, index=False)

print('Saved:', summary_path)
print('Saved:', vol_path)